# Phase 10 — Deployment, MLOps, and packaging

This notebook trains and serializes the accepted lightweight outcome model, records a local MLflow experiment with data/version references, verifies deterministic loading and startup-failure behavior, and validates deployment/configuration artifacts. Conditional roadmap extensions are resolved from evidence: the measured linear model already wins; no reliable player/squad dataset exists; and synchronous simulation latency is below the async threshold. Therefore PyTorch, player embeddings, and a job queue are deliberately not added.

In [1]:
from pathlib import Path
import hashlib, json, os, time
import joblib, mlflow, numpy as np, pandas as pd, yaml
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd(); SEED=20260705; VERSION='outcome-logistic-1.0.0'
data_path=ROOT/'data/processed/matches_with_elo.csv'; data=pd.read_csv(data_path,parse_dates=['date']).query("date >= '2000-01-01'").sort_values(['date','match_id']).reset_index(drop=True)
features=['elo_difference','neutral']; X=data[features].astype(float); y=data.result.map({'H':0,'D':1,'A':2}); cut=int(len(data)*.85)
model=make_pipeline(StandardScaler(),LogisticRegression(C=.1,max_iter=1000,random_state=SEED)).fit(X.iloc[:cut],y.iloc[:cut]); probs=model.predict_proba(X.iloc[cut:]); metric=float(log_loss(y.iloc[cut:],probs,labels=[0,1,2]))
artifact_dir=ROOT/'artifacts'/VERSION; artifact_dir.mkdir(parents=True,exist_ok=True); model_path=artifact_dir/'model.joblib'; joblib.dump({'version':VERSION,'features':features,'model':model},model_path)
checksum=hashlib.sha256(model_path.read_bytes()).hexdigest(); data_hash=hashlib.sha256(data_path.read_bytes()).hexdigest(); manifest={'version':VERSION,'sha256':checksum,'data_sha256':data_hash,'trained_through':str(data.iloc[cut-1].date.date()),'holdout_log_loss':metric,'features':features}; (artifact_dir/'manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')

tracking=(ROOT/'mlflow.db').resolve(); mlflow.set_tracking_uri('sqlite:///'+tracking.as_posix()); mlflow.set_experiment('matchcast-production-candidate')
with mlflow.start_run(run_name=VERSION) as run:
    mlflow.log_params({'model':'logistic_regression','C':.1,'seed':SEED,'train_rows':cut,'features':','.join(features),'data_sha256':data_hash}); mlflow.log_metric('holdout_log_loss',metric); mlflow.log_artifact(str(artifact_dir/'manifest.json')); run_id=run.info.run_id

def load_versioned(path:Path,expected_version:str):
    if not path.exists(): raise RuntimeError(f'model artifact missing: {path}')
    payload=joblib.load(path)
    if payload.get('version')!=expected_version: raise RuntimeError('model version mismatch')
    return payload
loaded=load_versioned(model_path,VERSION); assert np.allclose(loaded['model'].predict_proba(X.iloc[cut:cut+10]),probs[:10])
try: load_versioned(ROOT/'artifacts/missing/model.joblib',VERSION); raise AssertionError('missing artifact accepted')
except RuntimeError as e: assert 'missing' in str(e)

required=[ROOT/'Dockerfile',ROOT/'docker-compose.yml',ROOT/'.env.example',ROOT/'.github/workflows/ci.yml',ROOT/'migrations/001_initial.sql',ROOT/'docs/architecture.md',ROOT/'docs/api.md',ROOT/'docs/model-card.md',ROOT/'docs/reproduction.md',ROOT/'docs/portfolio.md']
assert all(p.exists() for p in required); compose=yaml.safe_load((ROOT/'docker-compose.yml').read_text()); assert {'api','db'}<=set(compose['services']); assert 'healthcheck' in compose['services']['api'] and 'healthcheck' in compose['services']['db']
docker=(ROOT/'Dockerfile').read_text(); assert 'USER matchcast' in docker and 'HEALTHCHECK' in docker
workflow=yaml.safe_load((ROOT/'.github/workflows/ci.yml').read_text()); assert workflow['jobs']['verify']['steps']
env=(ROOT/'.env.example').read_text(); assert 'replace-with-a-local-secret' in env and not any(s in env for s in ['password123','secret123'])
decision={'pytorch':'deferred: accepted linear model is stronger than tested tree models; no evidence neural complexity improves 192-match backtest','player_embeddings':'deferred: repository has no reliable player/squad dataset','async_jobs':'deferred: bounded 1,000-run four-team simulations complete synchronously; add queue only after measured SLA breach','deployment':'local container specification complete; no external hosting target or credentials supplied'}
(ROOT/'reports/phase10_decisions.json').write_text(json.dumps(decision,indent=2),encoding='utf-8')
print({'version':VERSION,'holdout_log_loss':round(metric,4),'artifact_sha256':checksum[:12],'mlflow_run_id':run_id}); print('All Phase 10 artifact, tracking, loading, configuration, and deployment-manifest checks passed.')

2026/07/05 20:21:59 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/07/05 20:21:59 INFO mlflow.store.db.utils: Updating database tables


2026/07/05 20:22:00 INFO mlflow.tracking.fluent: Experiment with name 'matchcast-production-candidate' does not exist. Creating a new experiment.


{'version': 'outcome-logistic-1.0.0', 'holdout_log_loss': 0.8764, 'artifact_sha256': '5aa344ab6936', 'mlflow_run_id': '2ba2a73945cd43afac07ecc80b8b8ca6'}
All Phase 10 artifact, tracking, loading, configuration, and deployment-manifest checks passed.
